# Speech Recognition

Ziele der heutigen Übung:
- Kennenlernen der SpeechRecognition Bibliothek
- Transkribieren einer Spracheingabe mit der Google Web Speech API
- Verstehen von Erkennerergebnissen eines Spracherkenners
- Anbindung eines großen Sprachmodells (Large-Language Modells) zur Transkription
- Erleben von Herausforderungen der Spracherkennung


## Speech Recognition API

Heute lernen wir eine Bibliothek kennen, welche verschiedene Spracherkenner und online APIs anbindet. Wir verwenden hierfür die Google Speech API als Cloud Service und Vosk als Spracherkenner, welcher lokal mit begrenzten Ressourcen arbeitet.

Installiere folgende Bibliothek: https://pypi.org/project/SpeechRecognition/

In [ ]:
pip install SpeechRecognition

In [ ]:
import speech_recognition as sr
print(f"We use speech_recognition version: {sr.__version__}")

Um die Eingabe des Mikrofons aufzunehmen und an den Spracherkenner weiter zu leiten, nutzen wir die Bibliothek PyAudio. Installiere PyAudio mit pip. Je nach Betriebssystem kann es erforderlich sein zusätzlieh Pakete mit homebrew (macOS) oder apt (ubuntu) zu installieren. Du findest sicherlich mit einem LLM deiner Wahl oder Google eine Lösung:

In [ ]:
pip install pyaudio

In [ ]:
from pyaudio import PyAudio

p = PyAudio()
try:
    print(p.get_default_input_device_info())
except:
    print("No mics available")

## Transkribieren einer sprachlichen Äußerung mit der Google Web Speech API (cloud-basiert)

Folgendes Tutorial gibt gute Hinweise, wie wir die SpeechRecognition Bibliothek nutzen: https://www.simplilearn.com/tutorials/python-tutorial/speech-recognition-in-python 

In [ ]:
recognizer = sr.Recognizer()

Erfasse eine sprachliche Äußerung mit dem Mikrofon:

In [ ]:
with sr.Microphone() as source:
    print("Speak something...")
    audio_data = -1  # TODO: capture audio from mic
    print("Audio data recorded.")

Nun schicken wir die erfasste Äußerung an die Google Web Speech API. Sofern die API nicht erreichbar ist, wirft der ```recognizer``` einen ```RequestError```. Sofern in dem übergebenen ```audio_data``` keine sprachliche Äußerung enthalten ist gibt der ```recognizer``` einen ```UnknownValueError``` aus. Prüfe auf beide Exceptions und geben dem Nutzer ein entsprechendes Feedback auf der Konsole aus.

In [ ]:
try:
    text = ''  # TODO: send request to google cloud recognizer
    print("You said:", text)
except sr.RequestError:
    print("Error: Could not access Google Web Speech API;")
except sr.UnknownValueError:
    print("Sorry, I do not understand")


Neben Englisch kann die Google Web Speech API auch Deutsch. Passe den Code entsprechend an, dass Deutsch erkannt wird:

In [ ]:
try:
    text = ''  # TODO: change to German recognizer
    print("Du hast gesagt: ", text)
except sr.RequestError:
    print("Fehler: Google Web Speech API ist nicht erreichbar")
except sr.UnknownValueError:
    print("Sorry, Ich verstehe die Äußerung nicht")

Wir werden in der nächsten Vorlesung genauer auf die Technik der Spracherkennung schauen. Heute möchten wir uns jedoch bereits mehrer Erkennerergebnisse und die jeweilige Konfidenz anschauen. Mit dem Parameter ```show_all``` gibt der ```recognizer``` mehrere Alternativen als JSON aus. Integriere dies in deine Erkennung (Tipp: für den weiteren Verlauf ist es ratsam dies als Funktion zu implementieren):

In [ ]:
import json

def recognize_google(audio_data: any):
    try:
        text = ''  # TODO: use show_all as parameter, you will get a JSON response, figure out how to get the text
        print(f"[Google] Recognition results: ", text)
    except sr.RequestError:
        print("[Google] Fehler: Google Web Speech API ist nicht erreichbar")
    except sr.UnknownValueError:
        print("[Google] Sorry, Ich verstehe die Äußerung nicht")
recognize_google(audio_data)

## Spracherkennung ohne Cloud und Internetanbindung

Gerade als Ingenieure haben wir oftmals mit einer fehlenden Internetverbindung in Produktionsumgebungen, begrenzten Hardware-Ressourcen, Datenschutzanforderungen oder was auch immer zu tun, das es verhindert Daten in die Cloud zu schicken. Für solche Fälle möchten wir uns mit https://alphacephei.com/vosk/ einen Spracherkenner anschauen, der einerseits lokal und andererseits auch mit begrenzen Ressourcen zurecht kommt (z.B. Raspberry PI). Installiere die Bibliothek mit ```pip```

In [ ]:
pip install vosk

Vosk unterstützt verschiedene Sprachen und bietet Sprachmodelle in unterschiedlicher Größe an: siehe hierzu https://alphacephei.com/vosk/models. Starte mit einem kleinen Modell (vosk-model-small-**), lade dieses herunter. 

Für den Download können wir das Shellscript ```sprc``` nutzen. Wir können dies direkt in einer Zelle dieses Jupyter Notebooks machen, indem wir eine Code Zelle erstellen und als Sprache Shell Script auswählen:

In [ ]:
! sprc download vosk --url <Add Modell URL>

Nutze die Funktion ```recognizer.recognize_vosk```, um deine Äußerung zu transkribieren. Auch hier ist es hilfreich dies als Funktion zu implementieren...


In [ ]:
def recognize_vosk(audio_data: any):
    # TODO implement similar to recognize_google(audio_data: any)
    pass
recognize_vosk(audio_data)

## Herausforderungen der Spracherkennung

Wir haben nun zwei Erkenner angebunden, einerseits ein großes Modell in der Cloud und andererseits ein kleines Modell, welches lokal erkennt. Doch wie gut funktionieren diese? Und wo sind die Unterschiede? Nehme dir etwas Zeit und versuche verschiedene Äußerungen aus, variiere dabei zum Beispiel:
- Anwendungsfälle
- Namen von Personen, Städten, Restaurants, ...
- Sprechgeschwindigkeit
- Lautstärke
- Deutlich vs. undeutliche Aussprache
- Dialekt (kann Google Fränkisch?)
- Kommandos vs. natürlichsprachlich
- Lange vs. kurze Äußerungen
- Englische Begriffe in einem deutschen Satz (oder anders rum)
- Umgangssprache

Speichere dir hierfür die mit dem Mikrofon aufgenommenen Daten in eine Variable ```audio_data``` und gebe diese Daten an ```recognize_google(audio_data, ..``` und ```recognize_vosk(audio_data, ..```. Hier ist es nun praktisch auf die zuvor erstellen Funktionen zurückzugreifen ;-)

In [ ]:
with sr.Microphone() as source:
    print("Speak something...")
    audio_data = recognizer.listen(source)
    print("Audio data recorded.")
    
    recognize_google(audio_data)
    recognize_vosk(audio_data)

## Vertiefung: Transkription und Übersetzung mit Faster Whisper

Neben Vosk und Google kann die SpechRecognition Library auch mit diversen anderen Sprachanbietern umgehen. Im entsprechenden GitHub Repo findest du eine Auflistung:
https://github.com/Uberi/speech_recognition/

Bei den Cloud Anbietern benötigt man in der Regel einen (kostenpflichtigen) API-Key. Interessant sind die OpenAI Whisper Modelle, die zusätzlich lokal laufen und recht vernünftige Ergebnisse liefern (eine Übersicht zu aktuellen Benchmarks liefert: https://huggingface.co/spaces/hf-audio/open_asr_leaderboard). Wegen der Modellgröße und Rechenkapazität möchten wir im folgenden ein faster_whisper Modell nutzen. Von der Implementierung sind größere Modelle jedoch identisch.

Installiere als erstes die notwendigen Packete:

In [ ]:
pip install faster_whisper; soundfile

Dann können wir auch schon ```recognize_faster_whisper(audio_data)``` nutzen, um unser Audiofile mit dem Faster-Whsiper base Modell zu transkribieren:

In [ ]:
# TODO: call faster whisper with audio_data

Mit dem Parameter ```show_dict=True``` bekommen wir noch einige interessanten Zusatzinformationen:

In [ ]:
# TODO: add parameter show_dict to faster whisper call

In [ ]:
print("Die Wahrscheinlichkeit, dass keine Sprache im Audiofile ist beträgt: ", # TODO: Output no_speech_prob)

Ebenso können wir über die ```init_options``` und ```transcribe_options``` weitere Parameter mitschicken und somit die Transkription von Whisper steuern. Viele der Parameter werden wir im Kapitel LLMs genauer betrachten. 

In [ ]:
from speech_recognition.recognizers.whisper_local.faster_whisper import InitOptionalParameters
from speech_recognition.recognizers.whisper_local.faster_whisper import TranscribeOptionalParameters

# see https://github.com/SYSTRAN/faster-whisper/blob/v1.1.0/faster_whisper/transcribe.py#L575
init_params = InitOptionalParameters()
init_params['device'] = 'cpu'  # gpu
init_params['num_workers'] = 5

# see https://github.com/SYSTRAN/faster-whisper/blob/v1.1.0/faster_whisper/transcribe.py#L692
transcribe_params = TranscribeOptionalParameters()
transcribe_params['task'] = 'translate'  # transcribe
transcribe_params['temperature'] = [0.0]
transcribe_params['word_timestamps'] = True

In [ ]:
# TODO: call faster whisper with init_params and transcribe_params. What is the difference?

In [ ]:
transcribe_params['temperature'] = [1.0]
# TODO: call faster whisper with new temperature parameter. What is the difference?